# AI Agent 基礎與記憶機制

## 目標

- 理解 AI Agent 的核心概念
- 建立你的第一個簡單 Agent
- 實作具有記憶功能的 Agent
- 使用 Session 管理對話上下文

---

## 環境設定

### 安裝必要套件

我們將使用 Google 的 Agent Development Kit (ADK) 來建構 Agent。

In [ ]:
# 安裝 Google ADK
!pip install -q --upgrade google-adk

### 設定 API 金鑰

您需要一個 Gemini API 金鑰才能使用本實驗。

**取得 API 金鑰的步驟：**

1. 前往 [Google AI Studio](https://aistudio.google.com/app/apikey)
2. 建立一個新的 API 金鑰
3. 在 Colab 左側選單中，點擊 🔑 圖示（Secrets）
4. 新增一個名為 `GOOGLE_API_KEY` 的 secret
5. 貼上您的 API 金鑰並儲存
6. 啟用該 secret 的存取權限

In [ ]:
import os
from google.colab import userdata

# 從 Colab Secrets 取得 API 金鑰
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ API 金鑰設定完成")
except Exception as e:
    print(f"❌ 錯誤：請確認您已在 Colab Secrets 中新增 'GOOGLE_API_KEY'")
    print(f"詳細資訊：{e}")

### 匯入必要模組

In [ ]:
import google.generativeai as genai
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner, Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

print("✅ 模組匯入成功")

---

## Part 1: 單純與 LLM 互動 - 體驗「無記憶」

在開始建立 Agent 之前，讓我們先體驗 LLM 的「無狀態」特性。

In [ ]:
# 建立一個「無狀態」的模型
model = genai.GenerativeModel("gemini-flash-latest")

# 第一次對話
user_message = "你好！我叫小明，我喜歡寫程式。"
print(f"👤 User: {user_message}")
response = model.generate_content(user_message)
print(f"🤖 LLM: {response.text}\n")

In [ ]:
# 第二次對話 - 測試記憶
user_message = "你還記得我叫什麼名字嗎？"
print(f"👤 User: {user_message}")
response = model.generate_content(user_message)
print(f"🤖 LLM: {response.text}\n")

**💡 觀察結果**

LLM 無法回答第二個問題！因為：
- ❌ 每次呼叫都是獨立的（stateless）
- ❌ LLM 看不到之前的對話內容
- ❌ 沒有「記憶」機制

**這就是為什麼我們需要 Agent！**

---

## Part 2: 什麼是 AI Agent？

### 2.1 概念介紹

**傳統的 LLM 互動模式：**

```
使用者輸入 → LLM → 回應
```

這種模式是**單向的**、**無狀態的**，LLM 只能根據當下的輸入產生回應。

**AI Agent 的互動模式：**

```
       🧠LLM
        ↓↑
使用者輸入 → Agent → 回應
        ↓↑
      💾記憶管理
```

**Agent 的核心組成：**
- 🧠 **LLM**：大腦，負責思考與決策
- 💾 **記憶**：對話歷史與狀態管理
- 🛠️ **工具**：手腳，能夠執行實際操作
- 🔄 **執行框架**：協調上述元素的運作

### 2.2 記憶的重要性

LLM 本身是**無狀態的**（stateless），這意味著：
- 它們只「看得到」當次的輸入
- 無法記住之前的對話內容
- 每次呼叫都是獨立的

💡 **比喻**：想像你和朋友聊天，但他每句話後都會忘記您說過什麼，這樣的對話會非常困難！

**這就是為什麼我們需要記憶機制：**
- ✅ Agent 系統負責管理對話歷史
- ✅ 每次呼叫 LLM 時，將歷史訊息一併傳入
- ✅ 實現「記憶」的效果

---

## Part 3: 建立第一個 Agent

### 3.1 建立一個簡單的 Agent

我們將使用 Google ADK 來建立一個簡單的 Agent。

In [ ]:
# 建立一個簡單的 Agent
simple_agent = Agent(
    name="simple_assistant",
    model=Gemini(model="gemini-flash-latest"),
    instruction="你是一個友善的助理，請用繁體中文回答使用者的問題。"
)

print("✅ Agent 建立完成")

### 3.2 使用 InMemoryRunner 執行 Agent

`InMemoryRunner` 是一個簡單的執行器，特點：
- ✅ 適合快速測試和原型開發
- ✅ 自動管理對話歷史（記憶功能）
- ✅ 不需要手動管理 Session

我們將使用 `run_debug()` 方法來快速測試 Agent。

In [ ]:
# 建立 Runner
runner = InMemoryRunner(agent=simple_agent)

# 第一次對話：介紹自己
response1 = await runner.run_debug("你好！我叫小明，我喜歡寫程式。")

In [ ]:
# 第二次對話：詢問個人資訊
response2 = await runner.run_debug("請問我叫什麼名字？我有什麼興趣？")

**Agent 記住了！但在實務現場，該如何實現記憶？**

**🤔 思考問題**
- 一個 Chat Bot 需要建立幾個 Agent？
- Agent 怎麼知道現在是哪位用戶在對話？

**💡 就像點餐機**
- **點餐機**：只有一台，所有人共用
- **號碼牌**：每位顧客有自己的編號
- **訂單**：用號碼牌找到對應的訂單記錄

不同使用者在同一台機器操作，但系統能用號碼牌找到正確的訂單！

---

## Part 4: Session - 短期記憶的關鍵

### 4.1 Session 的架構

```
應用程式
  └── 使用者
       ├── Session 1
       │    ├── Events (對話歷史)
       │    └── State (共享狀態)
       ├── Session 2
       │    ├── Events
       │    └── State
       └── Session 3
            ├── Events
            └── State
```

### 4.2 Session 管理的兩個關鍵元件

1. **Runner**：執行層
   - 管理使用者和 agent 之間的互動流程
   - 自動維護對話歷史
   - 處理上下文管理

2. **SessionService**：儲存層
   - 管理 session 的建立、儲存和讀取
   - 不同的實作適合不同需求（記憶體、資料庫）

---

## Part 5: 建立可以管理 Session 的 Agent

讓我們建立一個具有記憶功能，且可以管理不同 sessions 的 Agent。

### 5.1 概念釐清  
**Runner**（點餐機）包含：
- **Agent**: 點餐機的系統程式（有哪些菜單、支援哪些功能）
- **SessionService**: 訂單管理系統

| 概念 | 比喻 | 職責 |
|------|------|------|
| Runner | 點餐機本體 | 整個服務的執行環境 |
| Agent | 系統程式 | 定義能力（模型、工具、角色） |
| SessionService | 訂單管理系統 | 管理所有用戶的記憶 |
| User ID | 號碼牌 | 識別用戶身份 |
| Session | 個別訂單 | 單一用戶的對話記憶 |

In [ ]:
# 建立 Agent
session_agent = Agent(
    name="session_assistant",
    model=Gemini(model="gemini-flash-latest"),
    description="一個具有 Session 管理能力的助理",
    instruction="你是一個友善的助理，請用繁體中文回答使用者的問題。"
)

In [ ]:
# 建立 Runner
stateful_runner = Runner(
    agent=session_agent,
    app_name="session_management",
    session_service=InMemorySessionService()
)

print("✅ 具有 Session 管理能力的 Agent 建立完成")
print(f"    應用程式名稱：session_management")
print(f"    Session 服務：InMemorySessionService")

### 5.2 定義輔助函數來執行 session

In [ ]:
# 定義輔助函數來執行 session
async def run_session(runner, user_queries, user_id="default_user", session_name="default_session"):
    """
    執行一個對話 session

    參數：
        runner: Runner 實例
        user_queries: 字串或字串列表，包含使用者的查詢
        user_id: 識別 user
    """
    print(f"\n=== Session: {session_name} ===")

    # 將單一查詢轉換為列表
    if isinstance(user_queries, str):
        user_queries = [user_queries]

    # 建立或取得 session
    app_name = runner.app_name

    try:
        session = await runner.session_service.create_session(
            app_name=app_name,
            user_id=user_id,
            session_id=session_name
        )
    except:
        session = await runner.session_service.get_session(
            app_name=app_name,
            user_id=user_id,
            session_id=session_name
        )

    # 處理每個查詢
    for query in user_queries:
        print(f"\n👤 User：{query}")

        # 轉換為 ADK Content 格式
        content = types.Content(
            role="user",
            parts=[types.Part(text=query)]
        )

        # 執行並串流回應
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=content
        ):
            if event.content and event.content.parts:
                text = event.content.parts[0].text
                if text and text != "None":
                    print(f"🤖 Agent：{text}")

print("✅ 輔助函數定義完成")

### 5.3 實測記憶功能

In [ ]:
# 在同一個 session 中進行多輪對話
await run_session(
    stateful_runner,
    [
        "你好！我叫小明，我是一名資料科學家，喜歡研究機器學習。",
        "請問我的職業是什麼？",
        "我喜歡研究什麼領域？"
    ],
    user_id="user_007",
    session_name="session_test"
)

你應該會看到 Agent 能夠記住每輪對話中提供的資訊。

這是因為所有對話都在**同一個 session** 中，Runner 會自動維護對話歷史。

每個 session 的資料是獨立的，不同 session 之間不會共享資訊。讓我們驗證這一點：

In [ ]:
# 在新的 session 中詢問相同問題
await run_session(
    stateful_runner,
    ["請問我叫什麼名字？我的職業是什麼？"],
    user_id="user_007",
    session_name="new_session"
)

**💡 觀察**

在新的 session 中，Agent 無法回答這些問題，因為這是一個全新的對話。

這是因為 **session 的隔離性**：每個 session 都有自己獨立的對話歷史。

---

## 總結

### 🎉 恭喜！你已完成 AI Agent 基礎

在這個 Lab 中了解到：

✅ **AI Agent 的基本概念**
   - 理解 Agent 與傳統 LLM 的差異
   - 理解「記憶」是由 Agent 系統管理，而非 LLM

✅ **記憶機制的實作**
   - 體驗無狀態 LLM 的限制
   - 使用 Runner 自動管理對話歷史
   - 理解為什麼需要 Session 管理

✅ **Session 管理**
   - 使用 `InMemorySessionService` 進行快速開發
   - 理解 Session 的隔離性
   - 掌握 Runner 的角色與功能